# WM-811K Wafer Map Defect Classification (CNN)


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
import re

PKL_PATH = os.path.join(os.path.dirname(__file__), '..', 'LSWMD.pkl')
PROCESSED_PATH = os.path.join(os.path.dirname(__file__), 'processed_data.npz')
TARGET_SIZE = 64
DEFECT_TYPES = ['Center','Donut','Edge-Loc','Edge-Ring','Loc','Near-full','Random','Scratch']

class CompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith('pandas.indexes'):
            module = module.replace('pandas.indexes', 'pandas.core.indexes')
        return super().find_class(module, name)

def extract_label(x):
    if isinstance(x, np.ndarray):
        flat = x.flat[0] if x.size > 0 else 'unknown'
        return str(flat).strip()
    if isinstance(x, list):
        v = x[0] if len(x) > 0 else 'unknown'
        return str(v).strip()
    s = str(x).strip()
    if s in ('nan', 'None', ''):
        return 'unknown'
    m = re.findall(r"'([^']+)'", s)
    if m: return m[0]
    return s

def prepare_data():
    if os.path.exists(PROCESSED_PATH):
        print("Already preprocessed. Loading cached data.")
        return
    print("[1/3] Loading LSWMD.pkl...")
    with open(PKL_PATH, 'rb') as f:
        df = CompatUnpickler(f, encoding='latin1').load()
    print("[2/3] Extracting labels...")
    df['label'] = df['failureType'].apply(extract_label)
    df_cls = df[df['label'].isin(DEFECT_TYPES)].copy()
    print(f"Defective wafers: {len(df_cls)}")
    print("[3/3] Resizing to 64x64...")
    X_list, y_list = [], []
    for i, (_, row) in enumerate(df_cls.iterrows()):
        if i % 2000 == 0:
            print(f"  Progress: {i}/{len(df_cls)}")
        wmap = np.array(row['waferMap'], dtype=float)
        if wmap.size == 0 or wmap.shape[0] == 0:
            continue
        zy = TARGET_SIZE / wmap.shape[0]
        zx = TARGET_SIZE / wmap.shape[1]
        resized = zoom(wmap, (zy, zx), order=0)
        X_list.append(resized)
        y_list.append(row['label'])
    X = np.array(X_list)
    le = LabelEncoder()
    y = le.fit_transform(y_list)
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15/0.85, stratify=y_temp, random_state=42)
    np.savez_compressed(PROCESSED_PATH, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test, classes=le.classes_)
    print(f"Saved to {PROCESSED_PATH}")

class WaferDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X, self.y, self.augment = X, y, augment
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        img = self.X[idx].copy()
        if self.augment:
            if np.random.rand() > 0.5: img = np.fliplr(img)
            if np.random.rand() > 0.5: img = np.flipud(img)
            img = np.rot90(img, np.random.randint(0, 4))
        img = img / 2.0
        return torch.tensor(img, dtype=torch.float32).unsqueeze(0), torch.tensor(self.y[idx], dtype=torch.long)

def get_dataloaders(batch_size=64):
    prepare_data()
    data = np.load(PROCESSED_PATH, allow_pickle=True)
    train_d = WaferDataset(data['X_train'], data['y_train'], augment=True)
    val_d = WaferDataset(data['X_val'], data['y_val'])
    test_d = WaferDataset(data['X_test'], data['y_test'])
    return (DataLoader(train_d, batch_size=batch_size, shuffle=True, num_workers=0),
            DataLoader(val_d, batch_size=batch_size, num_workers=0),
            DataLoader(test_d, batch_size=batch_size, num_workers=0),
            data['classes'])

if __name__ == "__main__":
    prepare_data()



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    """Simple CNN for WM-811K wafer defect pattern classification.
    Input: (batch, 1, 64, 64) grayscale wafer maps
    Output: (batch, 8) logits for 8 defect classes
    """
    def __init__(self, num_classes=8):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = x.view(-1, 256 * 4 * 4)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

if __name__ == "__main__":
    model = SimpleCNN(num_classes=8)
    dummy = torch.randn(16, 1, 64, 64)
    out = model(dummy)
    print(f"Input: {dummy.shape}, Output: {out.shape}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")



In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.utils.class_weight import compute_class_weight
import importlib

dataset_module = importlib.import_module("01_dataset")
get_dataloaders = dataset_module.get_dataloaders

model_module = importlib.import_module("02_cnn_model")
SimpleCNN = model_module.SimpleCNN

EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 0.001
PATIENCE = 7
MODEL_SAVE_PATH = os.path.join(os.path.dirname(__file__), "best_cnn_model.pth")

def train_model():
    print("=" * 60)
    print("  WM-811K CNN Training")
    print("=" * 60)
    train_loader, val_loader, test_loader, classes = get_dataloaders(batch_size=BATCH_SIZE)
    num_classes = len(classes)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] {device}")
    model = SimpleCNN(num_classes=num_classes).to(device)
    y_train = train_loader.dataset.y
    class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_val_loss = float("inf")
    patience_counter = 0
    print(f"\n[Training] Total {EPOCHS} Epochs")
    for epoch in range(EPOCHS):
        model.train()
        train_loss = train_correct = train_total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        model.eval()
        val_loss = val_correct = val_total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.4f}")
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"  -> Improved! Saved: {MODEL_SAVE_PATH}")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"\n[Early Stopping] No improvement for {PATIENCE} epochs.")
                break
    print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}")

if __name__ == "__main__":
    train_model()



In [ ]:
import os
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import importlib

dataset_module = importlib.import_module("01_dataset")
get_dataloaders = dataset_module.get_dataloaders

model_module = importlib.import_module("02_cnn_model")
SimpleCNN = model_module.SimpleCNN

MODEL_SAVE_PATH = os.path.join(os.path.dirname(__file__), "best_cnn_model.pth")
RESULTS_DIR = os.path.join(os.path.dirname(__file__), "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

def evaluate_model():
    print("=" * 60)
    print("  WM-811K CNN Evaluation")
    print("=" * 60)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _, _, test_loader, classes = get_dataloaders(batch_size=64)
    num_classes = len(classes)
    model = SimpleCNN(num_classes=num_classes).to(device)
    if os.path.exists(MODEL_SAVE_PATH):
        model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
        print(f"[Model Loaded] {MODEL_SAVE_PATH}")
    else:
        print("[Warning] No trained model found. Using random weights.")
    model.eval()
    all_preds, all_labels = [], []
    print("Running inference on test set...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n[Results] Balanced Accuracy: {bal_acc*100:.2f}%")
    for cls in classes:
        r = report[cls]
        print(f"  {cls:12s}: F1={r['f1-score']:.3f} | Prec={r['precision']:.3f} | Rec={r['recall']:.3f}")
    fig, ax = plt.subplots(figsize=(11, 9))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt=".0%", cmap="Blues", ax=ax,
                xticklabels=classes, yticklabels=classes)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"CNN Confusion Matrix (Balanced Acc: {bal_acc*100:.2f}%)")
    plt.tight_layout()
    cm_path = os.path.join(RESULTS_DIR, "fig6_cnn_confusion_matrix.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {cm_path}")
    fig, ax = plt.subplots(figsize=(12, 6))
    f1_scores = [report[c]['f1-score'] for c in classes]
    ax.bar(classes, f1_scores, color="#00d2ff")
    ax.axhline(bal_acc, color="orange", linestyle="--", label=f"Balanced Acc={bal_acc*100:.1f}%")
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("F1-Score")
    ax.set_title("CNN F1-Score per Defect Pattern")
    ax.legend()
    plt.tight_layout()
    f1_path = os.path.join(RESULTS_DIR, "fig7_cnn_f1_scores.png")
    plt.savefig(f1_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {f1_path}")
    output = {"model": "SimpleCNN", "balanced_acc": float(bal_acc),
               "per_class": {c: {"f1": float(report[c]['f1-score']),
                                  "precision": float(report[c]['precision']),
                                  "recall": float(report[c]['recall'])} for c in classes}}
    json_path = os.path.join(RESULTS_DIR, "cnn_results.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"Saved: {json_path}")

if __name__ == "__main__":
    evaluate_model()

